<a href="https://colab.research.google.com/github/Autopilot19/projects/blob/main/multitask_chatbot_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🤖 Multi‑Task Chatbot (LangChain + Transformers) — Colab
**Before running:** In Colab, go to **Runtime → Change runtime type → GPU** for faster inference.

This notebook provides:
- Intent routing (chat, summarize, simplify by age, QA with context, translation EN↔FR, sentiment)
- LangChain chains around Hugging Face pipelines
- Gradio UI for easy interaction


In [ ]:

!pip -q install "transformers>=4.44.0" "accelerate>=0.33.0" "sentencepiece>=0.2.0" \
                 "langchain>=0.2.10" "langchain-huggingface>=0.0.3" gradio


In [ ]:

import re, torch
from typing import Dict, Any

from transformers import pipeline
from transformers.utils.logging import set_verbosity_error
set_verbosity_error()

from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough
from langchain_huggingface import HuggingFacePipeline

def pick_device():
    return 0 if torch.cuda.is_available() else -1

DEVICE = pick_device()
print("CUDA available:", torch.cuda.is_available(), "| device =", DEVICE)


CUDA available: True | device = 0


In [ ]:

MODELS = {
    "router": ("zero-shot-classification", "facebook/bart-large-mnli"),
    "chat": ("text2text-generation", "google/flan-t5-large"),
    "summarize": ("summarization", "facebook/bart-large-cnn"),
    "simplify": ("text2text-generation", "google/flan-t5-large"),
    "qa": ("question-answering", "deepset/roberta-base-squad2"),
    "translate_en_fr": ("translation", "Helsinki-NLP/opus-mt-en-fr"),
    "translate_fr_en": ("translation", "Helsinki-NLP/opus-mt-fr-en"),
    "sentiment": ("sentiment-analysis", "cardiffnlp/twitter-roberta-base-sentiment-latest"),
}


In [ ]:

print("⏳ Loading pipelines...")
pl_router = pipeline(*MODELS["router"], device=DEVICE)

pl_chat = pipeline(*MODELS["chat"], device=DEVICE, max_new_tokens=512)
pl_sum = pipeline(*MODELS["summarize"], device=DEVICE, max_length=256, min_length=40, do_sample=False)
pl_simplify = pipeline(*MODELS["simplify"], device=DEVICE, max_new_tokens=256)

pl_qa = pipeline(*MODELS["qa"], device=DEVICE)
pl_en_fr = pipeline(*MODELS["translate_en_fr"], device=DEVICE, max_new_tokens=256)
pl_fr_en = pipeline(*MODELS["translate_fr_en"], device=DEVICE, max_new_tokens=256)
pl_sent = pipeline(*MODELS["sentiment"], device=DEVICE)

# Wrap LLM-like pipelines in LangChain
llm_chat = HuggingFacePipeline(pipeline=pl_chat)
llm_sum = HuggingFacePipeline(pipeline=pl_sum)
llm_simplify = HuggingFacePipeline(pipeline=pl_simplify)

print("✅ Pipelines ready.")


⏳ Loading pipelines...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ Pipelines ready.


In [ ]:

chat_prompt = PromptTemplate.from_template(
    "Tu es un assistant utile, concis et précis. Réponds en français si l'utilisateur parle français.\n"
    "Question:\n{input}\n\nRéponse utile et directe:"
)

simplify_prompt = PromptTemplate.from_template(
    "Réécris et explique le texte ci-dessous pour qu’il soit compréhensible par un enfant de {age} ans. "
    "Utilise un langage simple, des phrases courtes et donne un exemple si utile.\n\nTexte:\n{input}\n\nVersion simplifiée:"
)

summary_prompt = PromptTemplate.from_template(
    "Résume clairement et factuellement le texte suivant en 5 à 7 phrases maximum.\n\nTexte:\n{input}\n\nRésumé:"
)


In [ ]:

TASK_LABELS = [
    "chat",
    "summarize",
    "simplify",
    "question_answering",
    "translate_en_to_fr",
    "translate_fr_to_en",
    "sentiment",
]

def detect_task(user_input: str) -> str:
    text = user_input.strip()

    # Quick heuristics
    if re.search(r"\b(resume|résume|résumé)\b", text, re.I):  # FR/EN
        return "summarize"
    if re.search(r"\b(simplify|simplifie|pour.*\b\d{1,2}\s*ans)\b", text, re.I):
        return "simplify"
    if "context:" in text.lower() and "question:" in text.lower():
        return "question_answering"
    if re.search(r"\btranslate to french\b|\b(en\s*->\s*fr)\b|\benglish to french\b", text, re.I):
        return "translate_en_to_fr"
    if re.search(r"\btranslate to english\b|\b(fr\s*->\s*en)\b|\bfrench to english\b", text, re.I):
        return "translate_fr_to_en"
    if re.search(r"\b(sentiment|opinion|positif|négatif|negative|positive)\b", text, re.I):
        return "sentiment"

    # Zero-shot fallback
    res = pl_router(text, candidate_labels=TASK_LABELS, multi_label=False)
    label = res.get("labels", ["chat"])[0]
    mapping = {
        "question_answering": "question_answering",
        "translate_en_to_fr": "translate_en_to_fr",
        "translate_fr_to_en": "translate_fr_to_en",
        "summarize": "summarize",
        "simplify": "simplify",
        "sentiment": "sentiment",
        "chat": "chat",
    }
    return mapping.get(label, "chat")

def extract_age(s: str, default=12):
    m = re.search(r"\bage\s*[:=]\s*(\d{1,2})\b", s, re.I) or re.search(r"\b(\d{1,2})\s*ans\b", s, re.I)
    return int(m.group(1)) if m else default

# Chains
chat_chain = chat_prompt | llm_chat
summ_chain = summary_prompt | llm_sum
simpl_chain = RunnableLambda(lambda x: {"input": x["input"], "age": extract_age(x["input"])}) | simplify_prompt | llm_simplify

def run_qa(x: Dict[str, Any]) -> str:
    text = x["input"]
    m_ctx = re.search(r"context\s*:\s*(.+?)\n+question\s*:\s*(.+)", text, re.I | re.S)
    if not m_ctx:
        return ("❗ Pour la tâche Q&A, fournis:\n"
                "context: <texte de référence>\nquestion: <ta question>\n")
    context, question = m_ctx.group(1).strip(), m_ctx.group(2).strip()
    ans = pl_qa(question=question, context=context)
    return f"🧠 Réponse: {ans.get('answer','(pas trouvé)')}\n(score: {ans.get('score',0):.3f})"

qa_chain = RunnableLambda(run_qa)

def tr_en_fr(x): return pl_en_fr(x["input"])[0]["translation_text"]
def tr_fr_en(x): return pl_fr_en(x["input"])[0]["translation_text"]
translate_en_fr_chain = RunnableLambda(tr_en_fr)
translate_fr_en_chain = RunnableLambda(tr_fr_en)

def run_sent(x):
    res = pl_sent(x["input"])[0]
    label = res.get("label", "").lower()
    score = res.get("score", 0.0)
    return f"🧐 Sentiment: {label} (confiance: {score:.3f})"

sentiment_chain = RunnableLambda(run_sent)

def branch_for(task: str):
    return {
        "summarize":      summ_chain,
        "simplify":       simpl_chain,
        "question_answering": qa_chain,
        "translate_en_to_fr": translate_en_fr_chain,
        "translate_fr_to_en": translate_fr_en_chain,
        "sentiment":      sentiment_chain,
        "chat":           chat_chain,
    }.get(task, chat_chain)

def run_multitask(user_text: str):
    task = detect_task(user_text)
    chain = branch_for(task)
    out = chain.invoke({"input": user_text}) if hasattr(chain, "invoke") else chain({"input": user_text})
    if isinstance(out, dict) and "text" in out:
        out = out["text"]
    return task, out


In [ ]:

HELP = """
Tâches supportées (auto‑détectées):
- Chat général (par défaut)
- Résumé: « résume ce texte: ... »
- Simplification: « simplifie pour 10 ans: ... » (ou ajoute "age:10")
- Q&A (exige contexte):
    context: <texte>
    question: <ma question>
- Traduction EN->FR: « translate to french: ... »
- Traduction FR->EN: « translate to english: ... »
- Sentiment: « sentiment de: ... »
/quit pour sortir, /help pour l’aide.
"""

print("🤖 Bot REPL. Tape /help pour l’aide.")
# Décommente pour utiliser en cell interactif
# while True:
#     user = input("\n👤 Toi: ").strip()
#     if user.lower() in ("/quit", "/exit"):
#         print("👋 Bye!"); break
#     if user.lower() in ("/help",):
#         print(HELP); continue
#     task, out = run_multitask(user)
#     print(f"🔎 Task: {task}\n🟦 {out}")


🤖 Bot REPL. Tape /help pour l’aide.


In [ ]:

import gradio as gr

HELP_TXT = """
**Tâches** (détection automatique) :
- Chat général (par défaut)
- **Résumé** : « résume ce texte: ... »
- **Simplification** : « simplifie pour 10 ans: ... » (ou `age:10`)
- **Q&A** : Inclure :
```
context: <texte de référence>
question: <votre question>
```
- **Traduction EN→FR** : « translate to french: ... »
- **Traduction FR→EN** : « translate to english: ... »
- **Sentiment** : « sentiment de: ... »
"""

def gradio_handle(user_text):
    if not user_text.strip():
        return "Veuillez entrer un texte."
    task, out = run_multitask(user_text)
    return f"**Task détectée**: `{task}`\n\n{out}"

with gr.Blocks() as demo:
    gr.Markdown("# 🤖 Multi‑Task Chatbot (LangChain + Transformers)")
    gr.Markdown(HELP_TXT)
    inp = gr.Textbox(label="Votre message", lines=8, placeholder="Ex: résume ce texte: ...")
    btn = gr.Button("Exécuter")
    out = gr.Markdown()
    btn.click(fn=gradio_handle, inputs=inp, outputs=out)

# Set share=True if you want a public link (beware limits in Colab)
demo.launch(share=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>